# DatasetBase: индекс и навигация по EEG-датасету

## Goal

Показать работу первого этапа реализации датасета: построение типизированного индекса без загрузки сигналов в память. После выполнения ноутбука мы сможем:

- получить `Sample` по ключам `(subject_id, trial_number, block_index)`;
- использовать вложенный `source_map`;
- проверить пути к паре EEG/EOG FIF;
- применять исключения субъектов и trials;
- подтвердить полноту корпуса `Data_Pattern`.

## Setup

Ноутбук использует локальный репозиторий и каталог `data/Data_Pattern`. FIF-файлы здесь только индексируются: MNE и массивы сигналов на этом этапе не загружаются.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Project root with pyproject.toml was not found")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import DatasetBase

DATASET_DIR = PROJECT_ROOT / "data" / "Data_Pattern"
STEP_TYPE = "patt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset source: {DATASET_DIR}")

Project root: /home/slauva/Projects/master-thesis-2024-2026/code
Dataset source: /home/slauva/Projects/master-thesis-2024-2026/code/data/Data_Pattern


### Key Assumptions

- `labels.json` является источником метаданных блоков.
- Для каждого блока обязательны оба файла: `patt_EEG_<index>.fif` и `patt_EOG_<index>.fif`.
- Порядок плоского индекса детерминирован: subject → trial → block.

## Steps

### 1. Построить индекс

Конструктор проверяет labels, уникальность block index и наличие EEG/EOG-файлов, но не читает содержимое FIF.

In [2]:
dataset = DatasetBase(DATASET_DIR, dataset_step_type=STEP_TYPE)

type_counts = Counter(sample.type for sample in dataset)
summary = pd.DataFrame(
    [
        {
            "samples": len(dataset),
            "subjects": len(dataset.source_map),
            "trials": sum(len(trials) for trials in dataset.source_map.values()),
            "geometric": type_counts["geometric"],
            "random": type_counts["random"],
        }
    ]
)
display(summary)

,samples,subjects,trials,geometric,random
0,540,33,60,360,180


### 2. Навигация по трем ключам

Один и тот же объект доступен через вложенную карту и через tuple-key.

In [3]:
first_sample = dataset.samples[0]
sample_key = (
    first_sample.subject_id,
    first_sample.trial_number,
    first_sample.block_index,
)

sample_from_map = dataset.source_map[sample_key[0]][sample_key[1]][sample_key[2]]
sample_from_key = dataset[sample_key]
assert sample_from_map == sample_from_key

sample_view = sample_from_key.model_dump(mode="json", by_alias=False)
sample_view["eeg_path"] = Path(sample_view["eeg_path"]).name
sample_view["eog_path"] = Path(sample_view["eog_path"]).name
pd.Series(sample_view, name="Sample")

subject_id                                                      1
trial_number                                                    1
block_index                                                     1
eeg_path                                           patt_EEG_1.fif
eog_path                                           patt_EOG_1.fif
img             [[0, 0, 0, 1, 0, 0], [0, 0, 1, 0, 0, 0], [0, 1...
type                                                    geometric
pattern_id                                                     10
Name: Sample, dtype: object

### 3. Посмотреть компактный срез индекса

Не выводим весь `source_map`: показываем только первые восемь записей.

In [4]:
index_preview = pd.DataFrame(
    [
        {
            "subject_id": sample.subject_id,
            "trial_number": sample.trial_number,
            "block_index": sample.block_index,
            "type": sample.type,
            "eeg_file": sample.eeg_path.name,
            "eog_file": sample.eog_path.name,
        }
        for sample in dataset.samples[:8]
    ]
)
display(index_preview)

,subject_id,trial_number,block_index,type,eeg_file,eog_file
0,1,1,1,geometric,patt_EEG_1.fif,patt_EOG_1.fif
1,1,1,2,geometric,patt_EEG_2.fif,patt_EOG_2.fif
2,1,1,3,geometric,patt_EEG_3.fif,patt_EOG_3.fif
3,1,1,4,geometric,patt_EEG_4.fif,patt_EOG_4.fif
4,1,1,5,geometric,patt_EEG_5.fif,patt_EOG_5.fif
5,1,1,6,geometric,patt_EEG_6.fif,patt_EOG_6.fif
6,1,1,7,random,patt_EEG_7.fif,patt_EOG_7.fif
7,1,1,8,random,patt_EEG_8.fif,patt_EOG_8.fif


### 4. Исключить отдельный trial

Пустой список исключает субъекта целиком, а список `Trial_*` — только выбранные trials.

In [5]:
dataset_without_trial = DatasetBase(
    DATASET_DIR,
    dataset_step_type=STEP_TYPE,
    exclude_samples={"S_1": ["Trial_2"]},
)

exclusion_check = pd.DataFrame(
    {
        "configuration": ["full", "without S_1/Trial_2"],
        "samples": [len(dataset), len(dataset_without_trial)],
    }
)
display(exclusion_check)
assert 2 not in dataset_without_trial.source_map[1]

,configuration,samples
0,full,540
1,without S_1/Trial_2,531


## Checks

Проверяем известные размеры корпуса, существование файлов и стабильный порядок индекса.

In [6]:
keys = [(sample.subject_id, sample.trial_number, sample.block_index) for sample in dataset]

assert len(dataset) == 540
assert type_counts == {"geometric": 360, "random": 180}
assert keys == sorted(keys)
assert all(sample.eeg_path.is_file() and sample.eog_path.is_file() for sample in dataset)
assert len(dataset_without_trial) == 531

pd.Series(
    {
        "expected_samples": "passed",
        "label_distribution": "passed",
        "stable_order": "passed",
        "all_fif_paths_exist": "passed",
        "trial_exclusion": "passed",
    },
    name="validation",
)

expected_samples       passed
label_distribution     passed
stable_order           passed
all_fif_paths_exist    passed
trial_exclusion        passed
Name: validation, dtype: str

## Next Steps

`DatasetBase` отвечает только за структуру и валидацию путей. В ноутбуке `1.1-numpy_dataset.ipynb` тот же индекс используется для ленивой загрузки EEG/EOG в NumPy через MNE.